<img src="https://res.cloudinary.com/dtizipxds/image/upload/q_auto/f_auto/v1776264397/banner_yvwajv.png" width="100%">


In [1]:
%pip install numpy pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


# Network Intrusion Detection - Preprocessing Exercise

This notebook prepares the intrusion dataset, trains a baseline model, and evaluates predictions on `test.csv` by comparing with `solution.csv`.


## 1) Imports and setup

Import required libraries and set notebook display/warning options.


In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

## 2) Load exercise files

Read `train.csv`, `test.csv`, and `solution.csv`, and verify they are available.


In [3]:
TRAIN_PATH = Path("train.csv")
TEST_PATH = Path("test.csv")
SOLUTION_PATH = Path("solution.csv")

for p in [TRAIN_PATH, TEST_PATH, SOLUTION_PATH]:
    assert p.exists(), f"Missing file: {p.resolve()}"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
solution_df = pd.read_csv(SOLUTION_PATH)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Solution shape: {solution_df.shape}")
train_df.head()


Train shape: (20153, 42)
Test shape: (5039, 41)
Solution shape: (5039, 42)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,http,SF,295,12771,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.0,0.00,0.00,1.00,0.00,0.00,255,255,1.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,normal
1,0,tcp,IRC,S1,197,2276,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,2,1,0.5,1.0,0.00,0.00,0.50,1.00,0.00,24,9,0.38,0.08,0.04,0.0,0.12,0.33,0.17,0.44,normal
2,0,tcp,efs,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,113,16,0.0,0.0,1.00,1.00,0.14,0.06,0.00,255,16,0.06,0.06,0.00,0.0,0.00,0.00,1.00,1.00,anomaly
3,0,udp,domain_u,SF,44,130,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,151,233,0.0,0.0,0.00,0.00,1.00,0.00,0.02,255,254,1.00,0.01,0.00,0.0,0.00,0.00,0.00,0.00,normal
4,0,tcp,http,SF,54540,8314,0,0,0,2,0,1,1,0,0,0,0,0,0,0,0,0,3,3,0.0,0.0,0.33,0.33,1.00,0.00,0.00,255,255,1.00,0.00,0.00,0.0,0.00,0.00,0.02,0.02,anomaly


## 3) Quick data quality check

Inspect data types, missing values, and duplicate rows in the training set.


In [4]:
print("Dtype counts:")
print(train_df.dtypes.value_counts())

print("\nMissing values (top 20):")
missing = train_df.isna().sum().sort_values(ascending=False)
print(missing.head(20))

print("\nDuplicate rows:", train_df.duplicated().sum())


Dtype counts:
int64      23
float64    15
object      4
Name: count, dtype: int64

Missing values (top 20):
duration                       0
dst_host_count                 0
srv_count                      0
serror_rate                    0
srv_serror_rate                0
rerror_rate                    0
srv_rerror_rate                0
same_srv_rate                  0
diff_srv_rate                  0
srv_diff_host_rate             0
dst_host_srv_count             0
protocol_type                  0
dst_host_same_srv_rate         0
dst_host_diff_srv_rate         0
dst_host_same_src_port_rate    0
dst_host_srv_diff_host_rate    0
dst_host_serror_rate           0
dst_host_srv_serror_rate       0
dst_host_rerror_rate           0
dst_host_srv_rerror_rate       0
dtype: int64

Duplicate rows: 0


## 4) Define target and label mapping

Use `class` as target and map labels to binary values: `normal -> 0`, `anomaly -> 1`.


In [5]:
# TODO: set the target column name
# TARGET_COL = "class"
TARGET_COL = ...
assert TARGET_COL in train_df.columns, f"Target column '{TARGET_COL}' not found in train.csv"
assert TARGET_COL in solution_df.columns, f"Target column '{TARGET_COL}' not found in solution.csv"
assert list(test_df.columns) == [c for c in solution_df.columns if c != TARGET_COL], "test.csv must match solution.csv without target"

# Normalize labels to binary: 1 for anomaly, 0 for normal.
y = (train_df[TARGET_COL].astype(str).str.strip().str.lower() != "normal").astype(int)

label_summary = pd.DataFrame({
    "count": y.value_counts().sort_index(),
    "ratio": y.value_counts(normalize=True).sort_index().round(4),
}, index=["normal(0)", "anomaly(1)"])
label_summary


,count,ratio
normal(0),NaN,NaN
anomaly(1),NaN,NaN


## 5) Separate features from target

Split the training dataframe into input features `X` and target `y`, and identify numeric/categorical columns.


In [6]:
X = train_df.drop(columns=[TARGET_COL])

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print("Categorical columns:", categorical_cols)


Numeric columns: 38
Categorical columns: 3
Categorical columns: ['protocol_type', 'service', 'flag']


## 6) Build preprocessing pipeline

Create transformations for numeric and categorical columns, then fit on a training split and transform validation/test sets.


In [7]:
# Compatibility helper for sklearn versions (sparse_output introduced in newer versions).
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_ohe()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ],
    remainder="drop",
)

# Internal validation split from train.csv
# TODO: split into fit/validation sets using train_test_split
# X_fit, X_val, y_fit, y_val = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )
X_fit, X_val, y_fit, y_val = ...

X_fit_prep = preprocessor.fit_transform(X_fit)
X_val_prep = preprocessor.transform(X_val)
# TODO: transform test features using the fitted preprocessor
# X_test_prep = preprocessor.transform(test_df)
X_test_prep = ...

print("Fit processed shape:", X_fit_prep.shape)
print("Validation processed shape:", X_val_prep.shape)
print("Test processed shape:", X_test_prep.shape)


Fit processed shape: (16122, 117)
Validation processed shape: (4031, 117)
Test processed shape: (5039, 117)


## 7) Train baseline model

Train logistic regression on preprocessed training data and evaluate on validation data.


In [8]:
# Baseline linear model for quick validation of preprocessing quality.
clf = LogisticRegression(max_iter=3000, n_jobs=-1)
clf.fit(X_fit_prep, y_fit)

val_pred = clf.predict(X_val_prep)
val_proba = clf.predict_proba(X_val_prep)[:, 1]

print("Validation Binary F1:", round(f1_score(y_val, val_pred), 4))
print("Validation ROC-AUC:", round(roc_auc_score(y_val, val_proba), 4))
print(classification_report(y_val, val_pred, target_names=["normal", "anomaly"]))


Validation Binary F1: 0.9701
Validation ROC-AUC: 0.9962
              precision    recall  f1-score   support

      normal       0.96      0.98      0.97      2152
     anomaly       0.98      0.96      0.97      1879

    accuracy                           0.97      4031
   macro avg       0.97      0.97      0.97      4031
weighted avg       0.97      0.97      0.97      4031



## 8) Package outputs and final exercise check

Store processed artifacts, predict on `test.csv`, and compare against `solution.csv` metrics.


In [9]:
# Optional: package transformed datasets for downstream modeling.
processed = {
    "X_fit": X_fit_prep,
    "X_val": X_val_prep,
    "X_test": X_test_prep,
    "y_fit": y_fit.to_numpy(),
    "y_val": y_val.to_numpy(),
    "feature_names": preprocessor.get_feature_names_out().tolist(),
}

print("Prepared keys:", list(processed.keys()))
print("Feature count:", len(processed["feature_names"]))

# Final exercise step: predict test.csv and compare with solution.csv
y_solution = (solution_df[TARGET_COL].astype(str).str.strip().str.lower() != "normal").astype(int)
# TODO: predict labels for the test set
# test_pred = clf.predict(X_test_prep)
test_pred = ...
test_proba = clf.predict_proba(X_test_prep)[:, 1]

print("\nTest vs Solution")
print("Test Binary F1:", round(f1_score(y_solution, test_pred), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_solution, test_proba), 4))
print(classification_report(y_solution, test_pred, target_names=["normal", "anomaly"]))


Prepared keys: ['X_fit', 'X_val', 'X_test', 'y_fit', 'y_val', 'feature_names']
Feature count: 117

Test vs Solution
Test Binary F1: 0.9673
Test ROC-AUC: 0.996
              precision    recall  f1-score   support

      normal       0.96      0.98      0.97      2690
     anomaly       0.98      0.96      0.97      2349

    accuracy                           0.97      5039
   macro avg       0.97      0.97      0.97      5039
weighted avg       0.97      0.97      0.97      5039

